# AI_AGG — Sales Data Summarization

`AI_AGG` produces a natural-language summary of grouped data — turning numbers into narratives.

| Item | Detail |
|---|---|
| **Function** | `SNOWFLAKE.CORTEX.AI_AGG` |
| **Data** | `SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM` (zero-ingestion) |
| **Use-case** | Auto-generated executive summaries from raw metrics |


## Step 1 — Environment Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Create Lineitem View

In [ ]:
CREATE OR REPLACE VIEW lineitem_sample AS
SELECT
    l_orderkey      AS order_key,
    l_quantity       AS quantity,
    l_extendedprice  AS extended_price,
    l_discount       AS discount,
    l_tax            AS tax,
    l_returnflag     AS return_flag,
    l_linestatus     AS line_status,
    l_shipdate       AS ship_date,
    l_shipmode       AS ship_mode
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM
WHERE l_shipdate BETWEEN '1995-01-01' AND '1995-12-31';

In [ ]:
SELECT COUNT(*) AS row_count FROM lineitem_sample;
SELECT * FROM lineitem_sample LIMIT 5;

## Step 3 — Summarize Revenue by Ship Mode

Use `AI_AGG` to generate a natural-language summary of the revenue breakdown.

In [ ]:
SELECT SNOWFLAKE.CORTEX.AI_AGG(
    'Summarize the revenue breakdown by shipping mode, highlighting the top performer',
    {
        'ship_mode': ARRAY_AGG(DISTINCT ship_mode),
        'total_revenue': SUM(extended_price),
        'avg_discount': ROUND(AVG(discount), 4),
        'order_count': COUNT(DISTINCT order_key)
    }
) AS summary
FROM lineitem_sample;

## Step 4 — Monthly Revenue Narrative

In [ ]:
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', ship_date)::DATE AS month,
        SUM(extended_price)                   AS revenue,
        COUNT(*)                               AS line_count
    FROM lineitem_sample
    GROUP BY month
    ORDER BY month
)
SELECT SNOWFLAKE.CORTEX.AI_AGG(
    'Provide a monthly revenue trend analysis for 1995, noting any seasonal patterns',
    {
        'months': ARRAY_AGG(month),
        'revenues': ARRAY_AGG(revenue),
        'line_counts': ARRAY_AGG(line_count)
    }
) AS monthly_summary
FROM monthly;

## Step 5 — Returns Analysis

In [ ]:
WITH returns_data AS (
    SELECT
        return_flag,
        COUNT(*)                AS item_count,
        SUM(extended_price)     AS total_value,
        ROUND(AVG(discount), 4) AS avg_discount
    FROM lineitem_sample
    GROUP BY return_flag
)
SELECT SNOWFLAKE.CORTEX.AI_AGG(
    'Analyze the return patterns — compare returned vs non-returned items by volume and value',
    {
        'return_flags': ARRAY_AGG(return_flag),
        'item_counts': ARRAY_AGG(item_count),
        'total_values': ARRAY_AGG(total_value),
        'avg_discounts': ARRAY_AGG(avg_discount)
    }
) AS returns_summary
FROM returns_data;

## Key Takeaways

- `AI_AGG` turns structured aggregates into readable executive summaries.
- Pass a prompt describing what insight you want + an OBJECT of aggregate values.
- Ideal for automated reporting, Slack/email digests, and dashboard annotations.
- Combine with scheduled tasks for daily/weekly narrative reports.
